# Evaluate an Italian base language model

This is the canonical, executable walkthrough of the complete library. Running the default **smoke** profile exercises all four evaluation families—LightEval benchmarks, BLiMP-IT minimal pairs, held-out perplexity, and controlled generation—using small limits. It then displays the resolved configuration, stage status, metrics with human-readable semantics, generation diagnostics, and the complete reproducibility bundle.

The notebook also provides **broad**, guarded **full**, and editable **custom** profiles. A smoke score proves that the pipeline works; it is not a publication result.


## 1. Select a GPU runtime

In Colab choose **Runtime → Change runtime type → GPU**. CPU works for tiny tests but is not suitable for most real models.

### What runs by default?

The main notebook defaults to `EVALUATION_PROFILE = "smoke"`: LightEval `quick` (1 task × at most 2 examples), at most 20 BLiMP-IT pairs, 3 streamed perplexity documents capped at 256 tokens each, and 3 prompts under every configured generation profile. This differs intentionally from the Python library's default `preset="quick"`, which skips optional LightEval and tests only the native components.

`None` means **unbounded within that field**; it never enables a disabled component. The full profile is blocked until you explicitly acknowledge it because it evaluates all 39 currently evaluable Italian LightEval variants, all BLiMP-IT subsets, the full selected perplexity corpus, and all generation prompts.


In [ ]:
# Hugging Face repo id, or a path under /content/drive after mounting Drive.
MODEL_SOURCE = "Gpeik/Sophira-360M-base"
TOKENIZER_SOURCE = "Gpeik/Sophira-360M-base"  # Empty also means: use MODEL_SOURCE.
MODEL_REVISION = None  # Automatically resolve the model repository's current commit SHA.
TOKENIZER_REVISION = None  # Resolve independently when TOKENIZER_SOURCE is another repository.
HF_TOKEN = ""  # Required only for gated/private Hub models.
MOUNT_GOOGLE_DRIVE = False
TRUST_REMOTE_CODE = False  # Enable only for a reviewed repository that requires custom model code.
TOKENIZER_USE_FAST = True
MAX_MODEL_LENGTH = None  # Optional context-length override.
ARTIFACT_SHA256 = None  # Recommended digest when MODEL_SOURCE is a local checkpoint.

# One switch controls the teaching workflow. Start with smoke.
# smoke: every evaluation family, tightly bounded.
# broad: all 39 evaluable Italian LightEval variants plus larger native samples.
# full: every available example/document/prompt; potentially many hours and large downloads.
EVALUATION_PROFILE = "smoke"  # smoke, broad, full, or custom
ALLOW_UNBOUNDED_FULL_RUN = False  # Must be True when EVALUATION_PROFILE == "full".

PROFILE_CONFIGS = {
    "smoke": dict(lighteval_suite="quick", lighteval_samples=2, blimp_samples=20,
                  ppl_subset="tiny", ppl_documents=3, ppl_tokens=256,
                  sequence_length=512, stride=256, generation_prompts=3),
    "broad": dict(lighteval_suite="all", lighteval_samples=10, blimp_samples=500,
                  ppl_subset="tiny", ppl_documents=100, ppl_tokens=1024,
                  sequence_length=1024, stride=512, generation_prompts=None),
    "full": dict(lighteval_suite="all", lighteval_samples=None, blimp_samples=None,
                 ppl_subset="full", ppl_documents=None, ppl_tokens=None,
                 sequence_length=1024, stride=512, generation_prompts=None),
    # Edit this entry for an experiment-specific budget. None always means unbounded.
    "custom": dict(lighteval_suite="verified_windows", lighteval_samples=25, blimp_samples=1000,
                   ppl_subset="tiny", ppl_documents=250, ppl_tokens=1024,
                   sequence_length=1024, stride=512, generation_prompts=None),
}
if EVALUATION_PROFILE not in PROFILE_CONFIGS:
    raise ValueError(f"Unknown EVALUATION_PROFILE: {EVALUATION_PROFILE}")
PROFILE = PROFILE_CONFIGS[EVALUATION_PROFILE]

# Component switches. The default smoke profile executes all four families.
ENABLE_LIGHTEVAL = True
ENABLE_BLIMP_IT = True
ENABLE_PERPLEXITY = True
ENABLE_GENERATION = True
LIGHTEVAL_SUITE = PROFILE["lighteval_suite"]
MAX_LIGHTEVAL_SAMPLES = PROFILE["lighteval_samples"]  # Per selected LightEval task.
MAX_BLIMP_SAMPLES = PROFILE["blimp_samples"]  # Global cap across BLiMP-IT subsets.
MAX_PPL_DOCUMENTS = PROFILE["ppl_documents"]
MAX_PPL_TOKENS_PER_DOCUMENT = PROFILE["ppl_tokens"]
MAX_GENERATION_PROMPTS = PROFILE["generation_prompts"]
LIGHTEVAL_NUM_FEWSHOT_SEEDS = 1
LIGHTEVAL_DATASET_LOADING_PROCESSES = 1
LIGHTEVAL_EXTRA_ARGS = []  # Advanced raw LightEval CLI arguments.

BLIMP_DATASET_REPO = "NeTSlab/BLiMP-IT"
BLIMP_DATASET_SUBSET = None  # None discovers every linguistic subset.
BLIMP_SPLIT = "test"

# Perplexity source. Replace this with a genuinely held-out Italian corpus for research claims.
PPL_DATASET_REPO = "gsarti/clean_mc4_it"
PPL_DATASET_SUBSET = PROFILE["ppl_subset"]  # tiny, small, medium, large, or full.
PPL_DATASET_SPLIT = "validation"
PPL_DATASET_STREAMING = True  # Avoid materializing unused remote splits.
PPL_SEQUENCE_LENGTH = PROFILE["sequence_length"]
PPL_STRIDE = PROFILE["stride"]
PPL_PRESERVE_DOCUMENT_BOUNDARIES = True
PPL_ADD_BOS_TOKEN = True
PPL_ADD_EOS_TOKEN = False
PPL_PER_DOCUMENT_STATS = True

GENERATION_PROFILES = [
    {"name": "greedy", "do_sample": False, "max_new_tokens": 128},
    {"name": "temp_0_7_top_p_0_9", "do_sample": True, "temperature": 0.7, "top_p": 0.9, "max_new_tokens": 128},
    {"name": "temp_0_8_top_p_0_95", "do_sample": True, "temperature": 0.8, "top_p": 0.95, "max_new_tokens": 128},
]

OVERWRITE_RESULTS = False  # False resumes/skips completed stages for the same resolved run.
SAVE_DETAILS = True  # Preserve sample-level LightEval outputs; consumes more storage.
OUTPUT_ROOT = "evaluation_results"
RANDOM_SEED = 13

MODEL_DEVICE = None  # None selects CUDA when available.
MODEL_DTYPE = "auto"
MODEL_BATCH_SIZE = 1
PARALLELISM = "auto"  # Replicates the model across all visible GPUs when launched below.
NUM_PROCESSES = "auto"  # Or set an explicit visible-GPU count.


In [ ]:
import os
import shutil
from pathlib import Path

if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

%cd /content
repo_path = Path("/content/it_eval_autoregressive_llms")
if repo_path.exists():
    shutil.rmtree(repo_path)
!git clone https://github.com/GiorgosPeikos/it_eval_autoregressive_llms.git
%cd /content/it_eval_autoregressive_llms


In [ ]:
!python -m pip install --upgrade "pip<27" "setuptools<82" wheel
!python -m pip install "lighteval[multilingual]==0.13.0" --no-deps
!python -m pip install -r constraints/lighteval-python310-313.txt
!python -m pip install -e . --no-deps


In [ ]:
import torch
from it_eval_framework.utils.lighteval_runtime import lighteval_environment_report

if ENABLE_LIGHTEVAL:
    lighteval_report = lighteval_environment_report()
    print(f"LightEval preflight: {lighteval_report}")
    if lighteval_report["errors"]:
        raise RuntimeError("LightEval preflight failed; rerun the installation cell.")

selected_device = MODEL_DEVICE or ("cuda" if torch.cuda.is_available() else "cpu")
print(f"Selected device: {selected_device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Warning: no GPU detected. Use a Colab GPU runtime for real models.")


In [ ]:
from pathlib import Path
import yaml

if EVALUATION_PROFILE == "full" and not ALLOW_UNBOUNDED_FULL_RUN:
    raise RuntimeError("The full profile is unbounded. Set ALLOW_UNBOUNDED_FULL_RUN=True after reading the warning above.")

config = {
    "run_name": "colab_model_eval",
    "model": {
        "source": MODEL_SOURCE,
        "revision": MODEL_REVISION,
        "tokenizer_source": TOKENIZER_SOURCE or MODEL_SOURCE,
        "tokenizer_revision": TOKENIZER_REVISION,
        "device": selected_device,
        "dtype": MODEL_DTYPE,
        "batch_size": MODEL_BATCH_SIZE,
        "trust_remote_code": TRUST_REMOTE_CODE,
        "tokenizer_use_fast": TOKENIZER_USE_FAST,
        "max_model_length": MAX_MODEL_LENGTH,
        "artifact_sha256": ARTIFACT_SHA256,
    },
    "output": {"root_dir": OUTPUT_ROOT, "overwrite": OVERWRITE_RESULTS, "save_details": SAVE_DETAILS},
    "runtime": {"seed": RANDOM_SEED, "parallelism": PARALLELISM, "num_processes": NUM_PROCESSES},
    "lighteval": {
        "enabled": ENABLE_LIGHTEVAL, "suite": LIGHTEVAL_SUITE if ENABLE_LIGHTEVAL else None,
        "max_samples": MAX_LIGHTEVAL_SAMPLES, "dataset_loading_processes": LIGHTEVAL_DATASET_LOADING_PROCESSES,
        "num_fewshot_seeds": LIGHTEVAL_NUM_FEWSHOT_SEEDS, "extra_args": LIGHTEVAL_EXTRA_ARGS,
    },
    "blimp_it": {
        "enabled": ENABLE_BLIMP_IT, "dataset_repo": BLIMP_DATASET_REPO, "dataset_subset": BLIMP_DATASET_SUBSET,
        "split": BLIMP_SPLIT, "max_samples": MAX_BLIMP_SAMPLES,
        "dataset_revision": "4159ecb68388283488cb1d235a7e1946489bc62d",
    },
    "perplexity": {
        "enabled": ENABLE_PERPLEXITY, "dataset_repo": PPL_DATASET_REPO, "dataset_subset": PPL_DATASET_SUBSET,
        "dataset_revision": "167d5696e91ac89f17936f9d0059031cbc4c9e99",
        "dataset_trust_remote_code": True, "dataset_streaming": PPL_DATASET_STREAMING, "split": PPL_DATASET_SPLIT,
        "text_field": "text", "sequence_length": PPL_SEQUENCE_LENGTH, "stride": PPL_STRIDE,
        "max_documents": MAX_PPL_DOCUMENTS,
        "max_tokens_per_document": MAX_PPL_TOKENS_PER_DOCUMENT,
        "preserve_document_boundaries": PPL_PRESERVE_DOCUMENT_BOUNDARIES,
        "add_bos_token": PPL_ADD_BOS_TOKEN, "add_eos_token": PPL_ADD_EOS_TOKEN,
        "per_document_stats": PPL_PER_DOCUMENT_STATS,
    },
    "generation": {
        "enabled": ENABLE_GENERATION, "prompts_path": "configs/generation_prompts.yaml",
        "max_prompts": MAX_GENERATION_PROMPTS, "seed": RANDOM_SEED, "profiles": GENERATION_PROFILES,
    },
}
config_path = Path("configs/colab_model_eval.yaml")
config_path.write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False), encoding="utf-8")
print(config_path.read_text(encoding="utf-8"))


## 2. Inspect the generated YAML, then run

The settings cell is converted into the same validated YAML schema used by the CLI and Python API. The runner resolves mutable Hugging Face revisions to immutable commit SHAs, prints every enabled component and LightEval task, and saves the expanded configuration as `resolved_config.yaml`.

Parameter scopes matter: `lighteval.max_samples` is a cap **per task**; `blimp_it.max_samples` is a cap **across all BLiMP subsets**; `perplexity.max_documents` is a document cap; `max_tokens_per_document` truncates each selected document; and `generation.max_prompts` is multiplied by the number of decoding profiles. Dataset preparation may still download a complete source split before a sample cap can be applied.

### Configuration map

| Block | Important parameters | Meaning |
|---|---|---|
| `model` | `source`, `revision` | Hub ID/local checkpoint and immutable model snapshot |
| `model` | `tokenizer_source`, `tokenizer_revision` | Independently resolved tokenizer; defaults to model source |
| `model` | `dtype`, `device`, `batch_size` | Weight precision, placement, and per-process LightEval batch size |
| `model` | `trust_remote_code` | Executes repository model code; use only after review |
| `model` | `max_model_length`, `artifact_sha256` | Context override and local-artifact identity |
| `output` | `overwrite`, `save_details` | Recompute completed stages and preserve sample-level details |
| `runtime` | `seed` | Reproducible sampling/base seed |
| `runtime` | `parallelism`, `num_processes` | Single process, replicated multi-GPU inference, or model sharding controls |
| `lighteval` | `suite`, `task_aliases` | Select predefined or explicit benchmark tasks |
| `lighteval` | `max_samples` | Per-task inference cap; `None` means every example |
| `lighteval` | `num_fewshot_seeds` | Repeats few-shot selection across seeds where applicable |
| `blimp_it` | `dataset_subset`, `max_samples` | One linguistic subset or all; global pair cap |
| `perplexity` | `dataset_repo/subset/split` | Corpus identity; use genuinely held-out Italian text |
| `perplexity` | `sequence_length`, `stride` | Sliding context window and advance between windows |
| `perplexity` | `max_documents`, `max_tokens_per_document` | Global document and per-document token budgets |
| `perplexity` | BOS/EOS and boundary controls | Exact token stream and whether documents remain independent |
| `generation` | `prompts_path`, `max_prompts` | Controlled prompt collection and cap |
| `generation` | `profiles` | Greedy/sampling parameters such as temperature, top-p, and output length |

### Equivalent interfaces

This notebook launches validated YAML with `it-eval-launch`. The public CLI is `it-eval evaluate --model OWNER/MODEL --preset quick`, and the Python API is `evaluate(model="OWNER/MODEL", preset="quick")`. Component-specific `it-eval-run-*` commands are debugging interfaces; `it-eval-compare` compares compatible checkpoint summaries.


In [ ]:
!it-eval-launch --config configs/colab_model_eval.yaml


In [ ]:
from pathlib import Path
from IPython.display import Markdown, display
import json
import pandas as pd
from it_eval_framework.reporting.metric_semantics import annotate_metric_rows

run_configs = list(Path(OUTPUT_ROOT).rglob("run_config.yaml"))
if not run_configs:
    raise FileNotFoundError("No evaluation run was found. Run the evaluation cell first.")
run_dir = max(run_configs, key=lambda path: path.stat().st_mtime).parent
display(Markdown(f"**Complete result directory:** `{run_dir}`"))
resolved_config_path = run_dir / "resolved_config.yaml"
if resolved_config_path.exists():
    display(Markdown("### Resolved configuration (the configuration actually evaluated)"))
    display(Markdown(f"```yaml\n{resolved_config_path.read_text(encoding='utf-8')}\n```"))

state_path = run_dir / "run_state.json"
if state_path.exists():
    steps = json.loads(state_path.read_text(encoding="utf-8")).get("steps", {})
    status_rows = [{"stage": stage, **details} for stage, details in steps.items()]
    display(Markdown("### Stage status"))
    display(pd.DataFrame(status_rows).fillna("—"))

summary_path = run_dir / "summary.csv"
if not summary_path.exists():
    raise FileNotFoundError(f"The run has no summary yet: {summary_path}")
summary = pd.read_csv(summary_path)
component_labels = {"blimp_it": "BLiMP-IT", "perplexity": "Perplexity", "generation": "Generation", "lighteval": "LightEval"}
for component, metrics in summary.groupby("component", sort=False):
    display(Markdown(f"### {component_labels.get(component, component)} metrics"))
    visible = annotate_metric_rows(metrics).drop(columns=["component"]).dropna(axis=1, how="all").reset_index(drop=True)
    display(visible)
    if visible["metric"].astype(str).str.contains("acc|exact_match|f1", case=False, regex=True).any():
        display(Markdown("A value of **1.0** on a 0–1 accuracy-like metric means every evaluated item received a score of 1 under that metric; **0.75 means 75%**. Always read `sample_count`: 1.0 on 2 smoke examples is not evidence of perfect dataset-wide performance."))

report_path = run_dir / "report.md"
if report_path.exists():
    display(Markdown("### Complete metric report"))
    display(Markdown(report_path.read_text(encoding="utf-8")))

generations_path = run_dir / "generations.jsonl"
if generations_path.exists():
    generation_rows = [json.loads(line) for line in generations_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    preview = pd.DataFrame([{
        "prompt_id": row.get("prompt_identifier"),
        "profile": row.get("decoding_profile", {}).get("name"),
        "prompt": row.get("prompt_text"),
        "generated_text": row.get("generated_text", "")[:500],
        "words": row.get("output_length_words"),
        "distinct_2": row.get("distinct_2"),
        "repeated_3gram_rate": row.get("repeated_3gram_rate"),
        "unfinished": row.get("unfinished_output"),
        "artifacts": row.get("artifact_flags"),
    } for row in generation_rows[:6]])
    display(Markdown(f"### Generation preview ({len(preview)} of {len(generation_rows)})"))
    with pd.option_context("display.max_colwidth", 500):
        display(preview)


In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive("italian_model_evaluation", "zip", root_dir=run_dir)
files.download(archive)


## 3. How to interpret the results

### LightEval

Each row is one metric for one resolved task (and, where applicable, one subject). Multiple-choice accuracy is the mean of a per-example correct/incorrect indicator: the model scores the available choices, the highest-scoring choice is selected, and the item receives 1 if that choice is gold and 0 otherwise. Thus accuracy `1.0` means all **evaluated** items were correct—not that the model is universally correct. `cf`, `mcf`, and `hybrid` are different prompt/scoring formulations and must not be averaged together without an explicit research rationale. Exact match requires the normalized generated answer to match an accepted reference; F1 gives partial credit for overlap. `stderr` estimates sampling uncertainty; roughly `value ± 1.96 × stderr` is an approximate 95% interval when that approximation is appropriate.

### BLiMP-IT

For each minimal pair, the framework sums next-token log-probabilities for the grammatical and ungrammatical sentence. The pair is correct when the grammatical sentence has at least the ungrammatical sentence's score. Overall accuracy is `correct pairs / evaluated pairs`; phenomenon rows use the same formula within that subset. Accuracy `1.0` means every evaluated pair was preferred correctly. It measures grammatical preference under total sentence likelihood, not general language quality.

### Perplexity

Mean loss is total negative log-likelihood divided by scored target tokens. Token perplexity is `exp(mean loss)`. Lower is better; perplexity is not a percentage and `1.0` is the theoretical perfect-prediction limit. Compare perplexity only with the same model tokenizer, corpus, split, normalization, sequence length, stride, boundary policy, and token budget. A corpus seen during training invalidates a held-out interpretation.

### Controlled generation

Generation has no single quality score. Inspect text across every prompt and decoding profile. `distinct_n` is the fraction of unique n-grams (higher often means more lexical diversity, but is not automatically better); `repeated_3gram_rate` measures repeated trigrams (lower usually indicates less degeneration); `unfinished_output` is a punctuation heuristic; and artifact flags detect encoding/repetition symptoms. Human review remains necessary for coherence, factuality, safety, and Italian quality.

### What to retain and report

Keep the downloaded ZIP. Report the repository commit, resolved model/tokenizer revisions, exact suite/task aliases, few-shot setting, sample/document/token limits, dataset revisions and splits, device/dtype, and whether details were saved. Compare checkpoints only when these evaluation settings match. The complete formulas, file map, and reporting cautions are in `docs/RESULTS_README.md` and `docs/TASK_REFERENCE.md`.
